In [ ]:
import xarray as xr
import fsspec
import pandas as pd
import holoviews as hv
import panel as pn
import echoshader
import numpy as np
import warnings

# Suppress warnings (optional)
warnings.filterwarnings('ignore')

# Initialize Extensions
hv.extension('bokeh')
pn.extension()

# Define dimensions
time_dim = hv.Dimension('ping_time', label='Time')
depth_dim = hv.Dimension('depth', label='Depth (m)')

# S3 Connection details
S3_URL = "s3://agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_8_wt_20170731_180848_f0002.zarr"
storage_options = {"endpoint_url": "https://sdsc.osn.xsede.org"}
fs = fsspec.filesystem("s3", anon=True, client_kwargs=storage_options)

print("✅ Cell 1 complete - All imports loaded")
print(f"   HoloViews version: {hv.__version__}")
print(f"   Panel version: {pn.__version__}")
print(f"   Echoshader version: {echoshader.__version__}")

In [ ]:
import pandas as pd
import numpy as np

# Create sample hake school regions in Echoregions standard format
# Format: region_id (int), time (datetime64[ns] array), depth (float array)

sample_df = pd.DataFrame({
    'region_id': [1, 2, 3, 4],
    'time': [
        # Region 1: 18:10 - 18:15
        np.array([
            '2017-07-31T18:10:00',
            '2017-07-31T18:15:00',
            '2017-07-31T18:15:00',
            '2017-07-31T18:10:00'
        ], dtype='datetime64[ns]'),
        
        # Region 2: 18:20 - 18:25
        np.array([
            '2017-07-31T18:20:00',
            '2017-07-31T18:25:00',
            '2017-07-31T18:25:00',
            '2017-07-31T18:20:00'
        ], dtype='datetime64[ns]'),
        
        # Region 3: 18:30 - 18:35
        np.array([
            '2017-07-31T18:30:00',
            '2017-07-31T18:35:00',
            '2017-07-31T18:35:00',
            '2017-07-31T18:30:00'
        ], dtype='datetime64[ns]'),
        
        # Region 4: 18:12 - 18:17
        np.array([
            '2017-07-31T18:12:00',
            '2017-07-31T18:17:00',
            '2017-07-31T18:17:00',
            '2017-07-31T18:12:00'
        ], dtype='datetime64[ns]')
    ],
    'depth': [
        np.array([40.0, 40.0, 80.0, 80.0]),      # Region 1 depth bounds
        np.array([100.0, 100.0, 150.0, 150.0]),  # Region 2 depth bounds
        np.array([50.0, 50.0, 90.0, 90.0]),      # Region 3 depth bounds
        np.array([60.0, 60.0, 100.0, 100.0])     # Region 4 depth bounds
    ]
})

print(f"✅ Loaded {len(sample_df)} hake school regions (Echoregions format)")
print(f"   Region IDs: {list(sample_df['region_id'].values)}")
print("\nDataFrame structure:")
print(sample_df)
print("\nData types:")
print(f"  region_id: {sample_df['region_id'].dtype}")
print(f"  time: {type(sample_df['time'].iloc[0])}, dtype={sample_df['time'].iloc[0].dtype}")
print(f"  depth: {type(sample_df['depth'].iloc[0])}, dtype={sample_df['depth'].iloc[0].dtype}")

In [ ]:
def parse_polygons_from_df(df):
    """
    Parse polygon boundaries from Echoregions-format DataFrame.
    
    Expected format:
    - region_id: int (unique identifier)
    - time: numpy array of datetime64[ns] values
    - depth: numpy array of float values
    
    Returns list of polygons in format compatible with HoloViews.
    """
    all_polygons = []
    
    for i, row in df.iterrows():
        try:
            region_id = row['region_id']
            times = row['time']   # numpy array of datetime64[ns]
            depths = row['depth']  # numpy array of float64
            
            # Validate data
            if not isinstance(times, np.ndarray) or not isinstance(depths, np.ndarray):
                print(f"⚠️ Skipping region {region_id}: time or depth is not numpy array")
                continue
                
            if len(times) != len(depths):
                print(f"⚠️ Skipping region {region_id}: time and depth lengths don't match")
                continue
                
            if len(times) < 3:
                print(f"⚠️ Skipping region {region_id}: need at least 3 points")
                continue
            
            # Convert datetime64[ns] to milliseconds (int64) for HoloViews compatibility
            # HoloViews expects numeric values, not datetime objects
            times_ms = times.astype('datetime64[ms]').astype(np.int64)
            
            # Ensure depths are float64
            depths_float = depths.astype(np.float64)
            
            polygon = {
                'ping_time': list(times_ms),
                'depth': list(depths_float),
                'region_id': f"Region_{region_id}"
            }
            
            all_polygons.append(polygon)
            
        except Exception as e:
            print(f"❌ Error parsing row {i} (region_id={row.get('region_id', 'unknown')}): {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if not all_polygons:
        print("❌ No valid polygons parsed!")
    else:
        print(f"✅ Parsed {len(all_polygons)} valid polygons")
        for poly in all_polygons:
            print(f"   - {poly['region_id']}: {len(poly['ping_time'])} points")
    
    return all_polygons

print("✅ Cell 3 complete - Parser function defined (Echoregions format)")

# Test the parser
print("\n" + "="*70)
print("TESTING PARSER")
print("="*70)
test_parsed = parse_polygons_from_df(sample_df)
if test_parsed:
    print(f"\n✅ Parser test successful!")
    print(f"   Example polygon structure:")
    print(f"   Region ID: {test_parsed[0]['region_id']}")
    print(f"   Time range: {test_parsed[0]['ping_time'][0]} to {test_parsed[0]['ping_time'][1]}")
    print(f"   Depth range: {min(test_parsed[0]['depth'])}m to {max(test_parsed[0]['depth'])}m")

In [ ]:
import xarray as xr
import panel as pn
import holoviews as hv
from holoviews.streams import PolyDraw, PolyEdit
import numpy as np
import pandas as pd
import fsspec
import warnings
import logging

warnings.filterwarnings('ignore')
logging.getLogger('root').setLevel(logging.ERROR)

pn.extension()


# CONFIGURATION & DATA LOADING
time_dim = hv.Dimension('ping_time', label='Time (ms)', value_format=lambda x: f'{x:.0f}')

S3_URL = "s3://agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_8_wt_20170731_180848_f0002.zarr"
fs = fsspec.filesystem("s3", anon=True, client_kwargs={"endpoint_url": "https://sdsc.osn.xsede.org"})

DATA_LOADED = False
ds_ooi = None

try:
    ds_ooi = xr.open_zarr(fs.get_mapper(S3_URL), consolidated=False)
    DATA_LOADED = True
    print("✅ S3 Connected")
except Exception as e:
    print(f"⚠️ S3 failed: {e}")


# PARSER & CACHING
def parse_polygons_from_df(df):
    results = []
    for _, row in df.iterrows():
        try:
            times = row['time']
            depths = row['depth']
            time_ms = times.astype('datetime64[ms]').astype(np.int64)
            results.append({
                'ping_time': time_ms.tolist(),
                'depth': depths.tolist(),
                'region_id': row['region_id']
            })
        except:
            pass
    return results

print("⏳ Pre-caching backgrounds...")
background_cache = {}

if DATA_LOADED and ds_ooi is not None:
    for region_id in sample_df['region_id']:
        current_df = sample_df[sample_df['region_id'] == region_id]
        parsed = parse_polygons_from_df(current_df)
        if parsed:
            try:
                time_values = np.array(parsed[0]['ping_time'])
                start_t = pd.to_datetime(time_values.min(), unit='ms')
                end_t = pd.to_datetime(time_values.max(), unit='ms')
                buffer = pd.Timedelta(minutes=5)
                ds_slice = ds_ooi.sel(ping_time=slice(start_t - buffer, end_t + buffer))
                
                if len(ds_slice.ping_time) > 0:
                    ds_slice = ds_slice.assign_coords({
                        'ping_time': (('ping_time',), ds_slice['ping_time'].data.astype('int64') // 10**6)
                    })
                    if 'depth' in ds_slice.dims:
                        ds_slice = ds_slice.rename({'depth': 'echo_range'})
                    
                    background = ds_slice.eshader.echogram(ds_slice.channel.values.tolist())()
                    background_cache[region_id] = background
                    print(f"  ✓ Region {region_id}")
            except:
                pass

print(f"✅ Cached {len(background_cache)} backgrounds")

# UI COMPONENTS

# Keep a backup of original data for Reset functionality
original_df = sample_df.copy()

region_ids = list(sample_df['region_id'])

# Mode selector
mode_selector = pn.widgets.RadioButtonGroup(
    name='Mode',
    options=['Browse', 'Edit'],
    value='Browse',
    button_type='primary',
    width=200
)

# Dropdown for region selection (better for many regions!)
region_dropdown = pn.widgets.Select(
    name='Select Region',
    options=region_ids,
    value=region_ids[0],
    width=200
)

# Navigation buttons
prev_btn = pn.widgets.Button(
    name='◀ Previous',
    button_type='light',
    width=95
)

next_btn = pn.widgets.Button(
    name='Next ▶',
    button_type='light',
    width=95
)

# Action buttons (initially hidden)
save_btn = pn.widgets.Button(
    name='💾 Save Changes',
    button_type='success',
    width=200
)

reset_btn = pn.widgets.Button(
    name='🔄 Reset Edits',
    button_type='warning',
    width=200
)

export_btn = pn.widgets.Button(
    name='📥 Export CSV',
    button_type='primary',
    width=200
)

# NEW: Load CSV button
load_btn = pn.widgets.FileInput(
    name='📂 Load CSV',
    accept='.csv',
    width=200
)

# Actions section (will be shown/hidden based on mode)
actions_section = pn.Column(
    pn.pane.Markdown("### Actions", styles={'font-size': '14px', 'color': '#666', 'margin-bottom': '5px'}),
    load_btn,
    pn.Spacer(height=5),
    save_btn,
    pn.Spacer(height=5),
    reset_btn,
    pn.Spacer(height=5),
    export_btn,
    visible=False  # Initially hidden!
)

# Status badge
status = pn.pane.Markdown(
    "🟢 **Browse Mode** - View Only",
    styles={
        'background': '#e8f5e9',
        'padding': '10px 15px',
        'border-radius': '20px',
        'border-left': '4px solid #4caf50',
        'margin': '10px 0'
    }
)


# STATE & LOGIC

poly_draw_stream = None

def update_nav(event):
    """Navigate between regions using Previous/Next buttons"""
    idx = region_ids.index(region_dropdown.value)
    if event.obj == next_btn:
        region_dropdown.value = region_ids[(idx + 1) % len(region_ids)]
    else:
        region_dropdown.value = region_ids[(idx - 1) % len(region_ids)]

prev_btn.on_click(update_nav)
next_btn.on_click(update_nav)

def on_mode_change(event):
    """Toggle between Browse and Edit modes"""
    is_edit = event.new == 'Edit'
    
    # Show/hide Actions section
    actions_section.visible = is_edit
    
    # Update status
    if is_edit:
        status.object = "🔴 **Edit Mode** - Click to add | Drag to move | Double-click to delete"
        status.styles = {
            'background': '#ffebee',
            'padding': '10px 15px',
            'border-radius': '20px',
            'border-left': '4px solid #f44336',
            'margin': '10px 0'
        }
    else:
        status.object = "🟢 **Browse Mode** - View Only"
        status.styles = {
            'background': '#e8f5e9',
            'padding': '10px 15px',
            'border-radius': '20px',
            'border-left': '4px solid #4caf50',
            'margin': '10px 0'
        }

mode_selector.param.watch(on_mode_change, 'value')

def save_changes(event):
    """Save edited polygon to DataFrame"""
    global poly_draw_stream, sample_df
    if poly_draw_stream is None:
        status.object = "⚠️ No edits to save"
        return
    
    try:
        data = poly_draw_stream.data
        if not data or len(data.get('xs', [])) == 0:
            return
        
        xs = data['xs'][0]
        ys = data['ys'][0]
        times_dt = pd.to_datetime(xs, unit='ms').values
        depths_arr = np.array(ys, dtype=np.float64)
        
        selected_id = region_dropdown.value
        idx = sample_df[sample_df['region_id'] == selected_id].index[0]
        sample_df.at[idx, 'time'] = times_dt
        sample_df.at[idx, 'depth'] = depths_arr
        
        status.object = f"✅ **Saved!** Region {selected_id} ({len(times_dt)} vertices)"
    except Exception as e:
        status.object = f"❌ Error: {e}"

def export_csv(event):
    """Export edited regions to CSV in Echoregions format"""
    try:
        # Create output filename with timestamp
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_filename = f"edited_regions_{timestamp}.csv"
        
        # Save to current working directory (accessible in Jupyter)
        output_path = output_filename
        
        # Convert DataFrame to CSV-compatible format
        export_data = []
        
        for _, row in sample_df.iterrows():
            region_id = row['region_id']
            times = row['time']  # datetime64[ns] array
            depths = row['depth']  # float64 array
            
            # Convert arrays to string representation for CSV
            # Format: "[value1, value2, value3, ...]"
            time_str = "[" + ", ".join([str(t) for t in times]) + "]"
            depth_str = "[" + ", ".join([str(d) for d in depths]) + "]"
            
            export_data.append({
                'region_id': region_id,
                'time': time_str,
                'depth': depth_str
            })
        
        # Create DataFrame and export
        export_df = pd.DataFrame(export_data)
        export_df.to_csv(output_path, index=False)
        
        status.object = f"✅ **Exported!** Saved as: {output_filename} (in current directory)"
        print(f"✅ CSV exported to: {output_path}")
        
    except Exception as e:
        status.object = f"❌ Export failed: {e}"
        print(f"❌ Export error: {e}")

def reset_edits(event):
    """Reset to original polygon by restoring from backup"""
    global sample_df, original_df
    
    try:
        selected_id = region_dropdown.value
        
        # Find the index for this region
        idx = sample_df[sample_df['region_id'] == selected_id].index[0]
        
        # Restore original data from backup
        sample_df.at[idx, 'time'] = original_df.at[idx, 'time'].copy()
        sample_df.at[idx, 'depth'] = original_df.at[idx, 'depth'].copy()
        
        status.object = f"🔄 **Reset** Region {selected_id} - Navigate away and back to see original"
        print(f"✅ Reset Region {selected_id} to original")
        
    except Exception as e:
        status.object = f"❌ Reset error: {e}"
        print(f"❌ Reset error: {e}")

def validate_loaded_regions(loaded_df, echogram_data):
    """
    Validate that loaded regions match the current echogram.
    Returns (is_valid, message)
    """
    try:
        # Get echogram time range
        echogram_start = pd.Timestamp(echogram_data.ping_time.min().values)
        echogram_end = pd.Timestamp(echogram_data.ping_time.max().values)
        
        # Check each region
        for idx, row in loaded_df.iterrows():
            region_id = row['region_id']
            times = row['time']
            
            # Convert to timestamps for comparison
            region_start = pd.Timestamp(times.min())
            region_end = pd.Timestamp(times.max())
            
            # Validation 1: Time range check
            if region_start < echogram_start or region_end > echogram_end:
                return False, f"Region {region_id} times ({region_start} to {region_end}) outside echogram range ({echogram_start} to {echogram_end})"
        
        return True, "Valid"
        
    except Exception as e:
        return False, f"Validation error: {e}"

def load_csv_file(event):
    """Load regions from uploaded CSV file with validation"""
    global sample_df, region_dropdown, ds_ooi
    
    if load_btn.value is None:
        status.object = "⚠️ No file selected"
        return
    
    try:
        # Read uploaded CSV
        import io
        csv_data = io.BytesIO(load_btn.value)
        loaded_df = pd.read_csv(csv_data)
        
        # Validation 1: Check required columns
        required_columns = ['region_id', 'time', 'depth']
        if not all(col in loaded_df.columns for col in required_columns):
            status.object = f"❌ Invalid CSV: Missing required columns. Need: {required_columns}"
            return
        
        # Parse string arrays back to numpy arrays
        for idx, row in loaded_df.iterrows():
            try:
                # Parse time string "[2017-07-31 18:10:00, ...]" back to numpy array
                time_str = row['time'].strip('[]')
                time_list = [t.strip() for t in time_str.split(',')]
                loaded_df.at[idx, 'time'] = np.array(time_list, dtype='datetime64[ns]')
                
                # Parse depth string "[40.0, 80.0, ...]" back to numpy array
                depth_str = row['depth'].strip('[]')
                depth_list = [float(d.strip()) for d in depth_str.split(',')]
                loaded_df.at[idx, 'depth'] = np.array(depth_list, dtype=np.float64)
                
            except Exception as e:
                status.object = f"❌ Parse error in row {idx}: {e}"
                return
        
        # Validation 2: Check time ranges against current echogram
        if DATA_LOADED and ds_ooi is not None:
            is_valid, message = validate_loaded_regions(loaded_df, ds_ooi)
            if not is_valid:
                status.object = f"❌ Validation failed: {message}"
                print(f"❌ Validation failed: {message}")
                return
        
        # All validation passed - load the data
        sample_df = loaded_df.copy()
        
        # Update region dropdown
        new_region_ids = list(sample_df['region_id'])
        region_dropdown.options = new_region_ids
        region_dropdown.value = new_region_ids[0]
        
        status.object = f"✅ **Loaded** {len(loaded_df)} regions from CSV"
        print(f"✅ Loaded {len(loaded_df)} regions successfully")
        
    except Exception as e:
        status.object = f"❌ Load error: {e}"
        print(f"❌ Load error: {e}")

save_btn.on_click(save_changes)
export_btn.on_click(export_csv)
reset_btn.on_click(reset_edits)
load_btn.param.watch(load_csv_file, 'value')  # Trigger when file is selected


# DASHBOARD

@pn.depends(region_dropdown.param.value, mode_selector.param.value)
def get_region_view(selected_id, mode):
    """Generate region view with cached backgrounds"""
    global poly_draw_stream
    
    current_df = sample_df[sample_df['region_id'] == selected_id]
    if current_df.empty:
        return pn.pane.Markdown("⚠️ No data")
    
    parsed = parse_polygons_from_df(current_df)
    if not parsed or selected_id not in background_cache:
        return pn.pane.Markdown("⚠️ No background")
    
    background = background_cache[selected_id]
    
    poly = hv.Polygons(
        [{'ping_time': r['ping_time'], 'echo_range': r['depth'], 'region_id': r['region_id']} 
         for r in parsed],
        kdims=[time_dim, hv.Dimension('echo_range', label='Depth (m)')],
        vdims=['region_id']
    ).opts(color='red', fill_alpha=0.3, line_width=2)
    
    if mode == 'Edit':
        # PolyDraw can do everything: add vertices, drag vertices, drag polygon, delete vertices
        poly_draw_stream = PolyDraw(
            source=poly, 
            drag=True,              # Can drag whole polygon
            num_objects=1,          # Only one polygon at a time
            show_vertices=True,     # Show vertex points
            vertex_style={'size': 10, 'color': 'red', 'fill_alpha': 0.8}
        )
    
    return background * poly

# Sidebar
sidebar = pn.Column(
    pn.pane.Markdown("## 🎛️ Controls", styles={'color': '#1976d2', 'margin-bottom': '15px'}),
    
    # Mode section
    pn.pane.Markdown("### Mode", styles={'font-size': '14px', 'color': '#666', 'margin-bottom': '5px'}),
    mode_selector,
    pn.Spacer(height=20),
    
    # Regions section with dropdown + navigation
    pn.pane.Markdown("### Regions", styles={'font-size': '14px', 'color': '#666', 'margin-bottom': '5px'}),
    region_dropdown,
    pn.Spacer(height=5),
    pn.Row(prev_btn, next_btn),
    pn.Spacer(height=20),
    
    # Actions section (conditionally visible)
    actions_section,
    
    styles={
        'background': '#f5f5f5',
        'padding': '20px',
        'border-radius': '8px',
        'box-shadow': '2px 0 5px rgba(0,0,0,0.1)'
    },
    width=250
)

# Main content
main_content = pn.Column(
    pn.pane.Markdown(
        "# 🐟 Hake School Region Browser",
        styles={'color': '#1976d2', 'margin-bottom': '5px'}
    ),
    pn.pane.Markdown(
        "*Modern sidebar interface with dropdown navigation*",
        styles={'color': '#666', 'font-size': '14px', 'margin-bottom': '15px'}
    ),
    status,
    get_region_view,
    styles={'padding': '20px'},
    sizing_mode='stretch_width'
)

# Full layout
layout = pn.Row(
    sidebar,
    main_content,
    styles={'background': 'white'},
    sizing_mode='stretch_width'
)

layout.show()

print("\n" + "="*70)
print("✅ IMPROVED SIDEBAR UI READY!")
print("="*70)
print("  ✅ Dropdown for region selection (scales to many regions)")
print("  ✅ Previous/Next navigation buttons")
print("  ✅ Actions section only visible in Edit mode")
print("  ✅ Fast navigation with cached backgrounds")
print("="*70)